In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd   # importing pandas libary for data cleaning ,analysis and manipulation of data
import torch           # importing torch library to load pytorch library



In [ ]:
# ----------------------------
# 1️⃣ Load Member-1 Cleaned Dataset for using it in pandas dadaframe
# ----------------------------

train_df = pd.read_csv("/content/drive/MyDrive/Fake News Detection /Data/train_dataset.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Fake News Detection /Data/test_dataset.csv")

print("Train shape:", train_df.shape)    # .shape define  no of rows , column (content , label)in train_dataset.csv
print("Test shape:", test_df.shape)       # .shape define  no of rows , column(content , label) in test_dataset.csv






Train shape: (35918, 2)
Test shape: (8980, 2)


In [ ]:
X_train = [str(x) for x in train_df['content'].tolist()]
# Extract all cleaned training news text prepared by Member-1 and convert them into a pure Python string list so transformer tokenizer can process them.
y_train = train_df['label']   #Extract training labels that correspond exactly to the training news articles.
#This extracts the label column. This column was created by Member-1 when they merged Fake.csv and True.csv.


X_test = [str(x) for x in test_df['content'].tolist()]
y_test = test_df['label']   #This extracts the correct labels for test data.

In [ ]:
from transformers import DistilBertTokenizerFast    # importing DistilBertTokenizerFast from transformers library

tokenizer = DistilBertTokenizerFast.from_pretrained(    # we will not train tokenizer from scratch instead of Download and load a pretrained tokenizer configuration
    "distilbert-base-uncased"                           # distilbert-base-uncases ---> model name of tokenizor
)


''' Computers cannot understand text like:
"Government announces free electricity"
They only understand numbers
So tokenizer converts text into:

1.input_ids
2.attention_mask   '''

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

' Computers cannot understand text like:\n"Government announces free electricity"\nThey only understand numbers\nSo tokenizer converts text into:\n\n1.input_ids\n2.attention_mask   '

In [ ]:
train_encodings = tokenizer(     #👉 tokenizer() function ko call kar rahe hain jo text ko numbers (tokens) me convert karta hai Output ko train_encodings variable me store kar rahe hain.
    X_train,                     #Training dataset ka text input tokenizer ko diya ja raha hai jo ki member 1 ka cleaned dataset se aya
    truncation=True,          #Agar koi news article bahut lamba hai (512 tokens se zyada) to tokenizer usko cut (truncate) kar dega.
                                #📌 Kyun? Transformer models ki maximum input length fixed hoti hai. DistilBERT → max 512 tokens.


    padding=True,           #Sab inputs ko same length ka banane ke liye padding apply ki ja rahi hai
    max_length=512,         #Maximum token length kitni hogi ...Transformer input ke liye maximum token size set kiya gaya hai
    return_tensors="pt"    # Output ko PyTorch tensor format me convert karne ke liye
)

# Now train_encodings contains: input_ids , attention_mask

test_encodings = tokenizer(    #Test dataset ko bhi same transformer format me convert kiya ja raha hai
    X_test,                     #same process for test dataset also
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors="pt"
)

In [ ]:
import torch   #PyTorch deep learning library ko import kar rahe hain.
               # Transformer model training aur tensor handling ke liye torch zaroori hai.

class NewsDataset(torch.utils.data.Dataset):   #Ek custom dataset class bana rahe hain. Ye PyTorch ke built-in Dataset class ko inherit kar raha hai.
                                              #Iska purpose: Tokenized news data ko model training ke format me dena

    def __init__(self, encodings, labels):  #Ye constructor hai.
                                             #Jab bhi dataset object create hoga: rain_dataset = NewsDataset(...)  Tab ye function automatically run hota hai.

        self.encodings = encodings         #Tokenizer ka output store kar rahe hain.
                                           #Ye hota hai: input_ids , attention_mask

        self.labels = labels.reset_index(drop=True)  #Labels ko store kar rahe hain.
                                                     # reset_index() isliye use karte hain: Train-test split ke baad index mismatch ho sakta hai ....Ye indexing ko clean sequence bana deta hai

    def __getitem__(self, idx):              #Jab model training ke time batch banata hai, tab ye function call hota hai.
                                             #Model bolta hai: "Mujhe sample number idx do"

        item = {key: val[idx] for key, val in self.encodings.items()}        # Ye dictionary comprehension hai.
                                                                              #Matlab: input_ids me se idx-th row lo .., attention_mask me se idx-th row lo
                                                                             # So ek single news article ka tokenized data milta hai.'''

        item['labels'] = torch.tensor(self.labels[idx])                  # Ab us news article ka correct label bhi add kar rahe hain.
                                                                             #So final sample ban gaya:{ 'input_ids' 'attention_mask' 'labels'}
                                                                            #Ye EXACT format HuggingFace Trainer ko chahiye.'''

        return item                                    #ye sample model ko return kar diya.

    def __len__(self):                                 #Ye batata hai dataset me total kitne samples hain.
        return len(self.labels)                        #Labels ka length = dataset ka length. # Dataset size equals number of labels

In [ ]:
train_dataset = NewsDataset(train_encodings, y_train)

# Yaha hum custom PyTorch Dataset object create kar rahe hain jo training ke liye use hoga.
#NewsDataset() class humne abhi define ki thi.
#Isme hum do cheeze pass kar rahe hain:

# 1️. train_encodings
#→ Ye tokenizer ka output hai
#→ Isme hota hai: input_ids ,attention_mask

#2️⃣ y_train
# Ye Member-1 ke dataset se aaya hua correct label column hai
# → Fake ya Real ka ground truth.

test_dataset  = NewsDataset(test_encodings, y_test)

#ye evaluation dataset object create kar raha hai.
# Model training ke baad isi dataset par performance check karega.

In [ ]:
tokenizer.save_pretrained("/content/drive/MyDrive/Fake News Detection /Data/Tokenizer")

## DistilBERT tokenizer object jisme tokenization configuration stored hai
#.save_pretrained() --  Ye HuggingFace ka built-in function hai.
#Iska kaam: Tokenizer configuration + vocabulary ko local folder me save karna

# "saved_tokenizer/" --  Ye folder name hai jahan tokenizer save hoga. Is folder ke andar automatically files banengi:

('/content/drive/MyDrive/Fake News Detection /Data/Tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/Fake News Detection /Data/Tokenizer/tokenizer.json')